In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import KFold, train_test_split
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ── Load raw full dataset ────────────────────────────────────
df_raw = pd.read_csv("train_raw.csv")

# Also load top50 train to get the feature names
df_top50_train = pd.read_csv("top50_train.csv")
top50_features = [c for c in df_top50_train.columns if c != "critical_temp"]

# ── Split raw data exactly as I did: 80/20, random_state=42
df_train_raw, _ = train_test_split(
    df_raw, test_size=0.2, random_state=42
)
print(f"Training samples for BOTH models: {len(df_train_raw)}")
# Should be 17,010 ✅

# ── Prepare features ────────────────────────────────────────
# All 81 features (for Shams)
X_81 = df_train_raw.drop("critical_temp", axis=1).values
y    = df_train_raw["critical_temp"].values

# Top 50 SHAP features only (for my model)
X_50 = df_train_raw[top50_features].values

print(f"my model input  : {X_50.shape}")
print(f"Shams model input : {X_81.shape}\n")

# ── Model definitions ────────────────────────────────────────
def build_my_model():
    base = [
        ('ridge', Ridge(alpha=1.0, solver='lsqr')),
        ('lasso', Lasso(alpha=0.000599, max_iter=1000)),
        ('knn',   KNeighborsRegressor(n_neighbors=5, weights='distance',
                                       algorithm='brute', metric='manhattan')),
        ('svr',   SVR(C=126.8554, kernel='rbf', gamma=0.194308,
                      epsilon=0.000139, tol=0.0001)),
        ('mlp',   MLPRegressor(hidden_layer_sizes=(200,100), activation='tanh',
                               learning_rate_init=0.005, solver='adam',
                               batch_size=256, alpha=0.0007,
                               max_iter=1000, random_state=42))
    ]
    meta = xgb.XGBRegressor(
        n_estimators=10000, learning_rate=0.008, max_depth=6,
        colsample_bytree=0.6, subsample=0.6, min_child_weight=5,
        gamma=0.1, reg_lambda=10.0, reg_alpha=0.5,
        tree_method='hist', random_state=42, n_jobs=-1, verbosity=0
    )
    return StackingRegressor(estimators=base, final_estimator=meta, cv=5)

def build_shams_model():
    base = [
        ('ridge', Ridge(alpha=0.001, solver='sparse_cg', tol=0.00001)),
        ('lasso', Lasso(alpha=0.001, tol = 0.0000001)),
        ('knn',   KNeighborsRegressor(n_neighbors=2, leaf_size=20,
                                       algorithm='brute', p=2)),
        ('svr',   SVR(C=100,cache_size=2000,coef0 =10,degree=1,
                      epsilon=1)),
        ('mlp',   MLPRegressor(hidden_layer_sizes=(56,32,25), activation='relu',
                               learning_rate_init=0.1,random_state=42))
    ]
    meta = RandomForestRegressor(n_estimators=120, min_samples_split=2,
                                  max_depth=20,random_state=42, n_jobs=-1)
    return StackingRegressor(estimators=base, final_estimator=meta, cv=5)

# ── 5-fold CV on SAME 17,010 samples, SAME folds ────────────
kf = KFold(n_splits=10, shuffle=True, random_state=42)

my_rmse_folds  = []
shams_rmse_folds = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_50)):
    print(f"Fold {fold+1}/10 ──────────────────────────────")

    # my MODEL — 50 features, StandardScaler
    X_tr_50,  X_val_50  = X_50[tr_idx],  X_50[val_idx]
    y_tr, y_val          = y[tr_idx],     y[val_idx]

    sc_my = StandardScaler()
    X_tr_50_sc  = sc_my.fit_transform(X_tr_50)
    X_val_50_sc = sc_my.transform(X_val_50)

    my_model = build_my_model()
    my_model.fit(X_tr_50_sc, y_tr)
    my_rmse = np.sqrt(np.mean((y_val - my_model.predict(X_val_50_sc))**2))
    my_rmse_folds.append(my_rmse)
    print(f"  my  RMSE: {my_rmse:.4f} K")

    # SHAMS MODEL — 81 features, MinMaxScaler, SAME fold indices
    X_tr_81,  X_val_81  = X_81[tr_idx],  X_81[val_idx]

    sc_shams = MinMaxScaler()
    X_tr_81_sc  = sc_shams.fit_transform(X_tr_81)
    X_val_81_sc = sc_shams.transform(X_val_81)

    shams_model = build_shams_model()
    shams_model.fit(X_tr_81_sc, y_tr)
    shams_rmse = np.sqrt(np.mean((y_val - shams_model.predict(X_val_81_sc))**2))
    shams_rmse_folds.append(shams_rmse)
    print(f"  Shams RMSE: {shams_rmse:.4f} K")
    print(f"  Diff      : {my_rmse - shams_rmse:+.4f} K\n")

# ── Paired t-test ────────────────────────────────────────────
my_arr  = np.array(my_rmse_folds)
shams_arr = np.array(shams_rmse_folds)
diffs     = my_arr - shams_arr

t_stat, p_value = stats.ttest_rel(my_arr, shams_arr)

print("\n" + "═"*55)
print("         PAIRED T-TEST RESULTS (FAIR COMPARISON)")
print("═"*55)
print(f"\n{'Fold':<6} {'my':>10} {'Shams':>10} {'Diff':>10}")
print(f"{'────':<6} {'────':>10} {'─────':>10} {'────':>10}")
for i in range(10):
    print(f"{i+1:<6} {my_arr[i]:>10.4f} {shams_arr[i]:>10.4f} {diffs[i]:>+10.4f}")
print(f"\n{'Mean':<6} {my_arr.mean():>10.4f} {shams_arr.mean():>10.4f} {diffs.mean():>+10.4f}")
print(f"{'Std':<6} {my_arr.std():>10.4f} {shams_arr.std():>10.4f}")
print(f"\n  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_value:.4f}")
print(f"  Mean RMSE reduction: {-diffs.mean():.4f} K")

if p_value < 0.01:
    print(f"\n  VERDICT: HIGHLY SIGNIFICANT (p < 0.01) ✅")
elif p_value < 0.05:
    print(f"\n  VERDICT: SIGNIFICANT (p < 0.05) ✅")
elif p_value < 0.10:
    print(f"\n  VERDICT: MARGINALLY SIGNIFICANT (p < 0.10) ⚠️")
else:
    print(f"\n  VERDICT: NOT SIGNIFICANT (p ≥ 0.10) ❌")



In [ ]:
# saving the results
pd.DataFrame({
    'Fold': range(1,11),
    'my_RMSE_K': my_arr,
    'Shams_RMSE_K': shams_arr,
    'Difference_K': diffs
}).to_csv("ttest_results_fair_.csv", index=False)
print("Saved: ttest_results_fair_.csv")

In [ ]:
from scipy import stats
import numpy as np

# my 10-fold RMSE values from the paired t-test run
# Replace these with my actual fold values
my_folds  = np.array([10.25835524,10.62351673,10.17224671,
                        10.1023187,10.25487063,9.697334377,
                        10.65969144,9.667316125,11.15332534,10.29177585

])  # my 10 RMSE values
shams_folds = np.array([10.35782162,10.7590356,10.7259306,
                        10.51406822,10.94141393,9.889641942,
                        11.44077419,10.1458736,11.02113116,10.76626399

])  # Shams 10 RMSE values

# Wilcoxon signed-rank test
stat, p_wilcoxon = stats.wilcoxon(my_folds, shams_folds, 
                                   alternative='less')
print(f"Wilcoxon signed-rank test:")
print(f"  statistic = {stat:.4f}")
print(f"  p-value   = {p_wilcoxon:.4f}")

# Also report how many folds my model won
wins = (my_folds < shams_folds).sum()
print(f"\nFolds where my model wins: {wins}/10")